# Old stuff

## Model 1

In [ ]:
pop_size = 100000
n_weeks = 800
SIR_obs = prop_to_nSIR(SIR(np.array([0.325]), np.array([0.055]), nt = n_weeks, N = pop_size, batch_size = 3), pop_size)
OR_hat = get_OR_hat(or_data, clade = "A", dataset = "NORM")
bsi_obs = col_to_BSI(SIR_obs, OR_hat = OR_hat, is_prop = False)#.reshape(-1,1)
#bsi_obs = (aggregate_BSI(bsi_obs), aggregate_BSI(bsi_obs), aggregate_BSI(bsi_obs))

def aggregate_BSI(bsi_obs, nan_locations  = [], batch_size=1):
    # Pick every 52th observation from bsi_obs
    return bsi_obs[:,::52]

def sum_over_bsi(bsi_obs, time_period = 52):
    # Take a sum over every ith week in bsi_obs (from i to i + time_period, where i is the current week)
    # Note: probably not applicable for proportion observations, only counts
    # time_period: time period to sum over. By default 52 (weeks).
    
    n_years = len(bsi_obs[0])//52
    #print(f'Number of years to sum: {n_years}')
    agg_bsi = []
    for i in range(0, n_years):
        start = i*time_period # which week to start summing at
        end = time_period*i + time_period # next week
        bsi_obs_yearly = bsi_obs[:,start:end]
        agg_bsi.append(np.sum(bsi_obs_yearly[:,], axis = 1))
    agg_bsi = np.asarray(agg_bsi).transpose()   
    
    return agg_bsi

def max_bsi_per_year(bsi_obs, time_period = 52):
    # Return the maximum number of BSI cases per year
    
    n_years = len(bsi_obs[0])//52
    agg_bsi = []
    
    for i in range(0, n_years):
        start = i*time_period # which week to start summing at
        end = time_period*i + time_period # next week
        agg_bsi.append(np.max(bsi_obs[:,start:end], axis = 1))
    agg_bsi = np.asarray(agg_bsi).transpose()  
    
    return agg_bsi


# test aggregate over summaries:
y = sum_over_bsi(bsi_obs)
print(y)
print(BSI_max_t(y))
print(BSI_max(y))
#print(BSI_k1(y))

In [ ]:
def BSI_max_t(y):
    # time to peak/maximum number of bsi cases
    # shaped (batch_size, n_obs)
    
    return np.argmax(y, axis = 1)

def BSI_max(y):
    # maximum number of BSI cases
    
    max_bsi = np.max(y[:,], axis = 1)
    
    return max_bsi


## Grid visualization

In [ ]:
# Alternative: evenly spaced parameter pairs
n = 3

inc = int(pairs.shape[0]/(n*n))

min_par1 = np.min(pairs[:,0])
min_par2 = np.min(pairs[:,1])

max_par1 = np.max(pairs[:,0])
max_par2 = np.max(pairs[:,1])

print(min_par2, min_par2)
print(max_par1, max_par2)

stepsize = pairs[3,0] - pairs[2,0]
print(stepsize)

# First, find 3 gamma values, evenly spaced:

gammas = [min(pairs[:,1]), min(pairs[:,1]) + (max(pairs[:,1]) - min(pairs[:,1]))/3, max(pairs[:,1])]
betas  = [min(pairs[:,0]), min(pairs[:,0]) + (max(pairs[:,0]) - min(pairs[:,0]))/3, max(pairs[:,0])]

print(gammas)
print(betas)

x = np.arange(0, 16)
y = []
par1s = []
par2s = []

for b in betas:
    for g in gammas:
        
        y.append(SIR_and_BSI_simulator(np.array([b]), np.array([g]), nt = n_weeks, N = pop_size,\
                                bsi_pars = bsi_pars, is_prop = True,\
                                is_agg = True, time_period = 52, batch_size = 1, random_state = None)[0])
        
        
        par1s.append(b)
        par2s.append(g)


        
print(y[1].shape)
# The following lines are a courtesy of ChatGPT :D
# Create a 3x3 grid of subplots
fig, axes = plt.subplots(3, 3, figsize=(10, 10))

x_tick_labels = [i + 1 for i in range(2001, 2017)]
print(x_tick_labels)
print(len(x_tick_labels))

y_min = 0
y_max = 0.03

# Flatten the axes array for easier iteration
axes = axes.flatten()

# Plot the line plots in each subplot
for i, ax in enumerate(axes):
    ax.plot(x, y[i])
    ax.set_title(f'Beta = {np.round(par1s[i],3)}, gamma = {np.round(par2s[i],4)}')
    ax.set_ylim(y_min, y_max)
    ax.set_xticks(x)
    ax.set_xticklabels(x_tick_labels, rotation = 90)
    
    point = ax.plot(BSI_max_t(y[i]), BSI_max(y[i]), "ro",label = "(BSI_max_t, BSI_max)")[0]
    ax.legend(handles = [point], loc='upper right')

# Adjust the spacing between subplots
plt.tight_layout()

# Display the grid of line plots
plt.show()

# Also show summaries
BSI_max(y[i])

sc = plt.scatter(pairs[:,0], pairs[:,1], c = dists, s = 1)

for i in range(0, 9):
    plt.scatter(par1s[i], par2s[i], c= "red", marker = "X")
plt.xlabel("Beta")
plt.ylabel("Gamma")
plt.colorbar(sc)
sc.set_cmap('viridis') # 'plasma'
#sc.set_clim(cutoff_lower, cutoff_upper)

In [ ]:
# Form a 3x3 grid of colonisation figures

import matplotlib.pyplot as plt
import numpy as np

# Generate some random data for the line plots
#x = np.linspace(0, n_weeks, )
x = np.arange(0, 16)



# Find nine parameter pair examples & visualize them in a grid

y = []

n = 3

inc = int(pairs.shape[0]/(n*n))

par1s = []
par2s = []

for i in range(0,9):
    
    if i == 0:
        j = i
    else:
        j += inc
    
    if j > pairs.shape[0]:
        j = pairs.shape[0] - 1

    print(j)
        
    y.append(SIR_and_BSI_simulator(np.array([pairs[j,0]]), np.array([pairs[j,1]]), nt = n_weeks, N = pop_size,\
                                bsi_pars = bsi_pars, is_prop = True,\
                                is_agg = True, time_period = 52, batch_size = 1, random_state = None)[0])
    
    par1s.append(pairs[j,0])
    par2s.append(pairs[j,1])
    
    
print(y[1].shape)
# The following lines are (mostly) a courtesy of ChatGPT :D
# Create a 3x3 grid of subplots
fig, axes = plt.subplots(3, 3, figsize=(10, 10))

x_tick_labels = [i + 1 for i in range(2001, 2017)]
print(x_tick_labels)
print(len(x_tick_labels))

y_min = 0
y_max = 0.025

# Flatten the axes array for easier iteration
axes = axes.flatten()

# Plot the line plots in each subplot
for i, ax in enumerate(axes):
    ax.plot(x, y[i])
    ax.set_title(f'Beta = {np.round(par1s[i],3)}, gamma = {np.round(par2s[i],4)}')
    ax.set_ylim(y_min, y_max)
    ax.set_xticks(x)
    ax.set_xticklabels(x_tick_labels, rotation = 90)
    
    point = ax.plot(BSI_max_t(y[i]), BSI_max(y[i]), "ro",label = "(BSI_max_t, BSI_max)")[0]
    ax.legend(handles = [point], loc='upper right')

# Adjust the spacing between subplots
plt.tight_layout()

# Display the grid of line plots
plt.show()

# Also show summaries
BSI_max(y[i])

sc = plt.scatter(pairs[:,0], pairs[:,1], c = dists, s = 1)

for i in range(0, 9):
    plt.scatter(par1s[i], par2s[i], c= "red", marker = "X")
plt.xlabel("Beta")
plt.ylabel("Gamma")
plt.colorbar(sc)
sc.set_cmap('viridis') # 'plasma'
#sc.set_clim(cutoff_lower, cutoff_upper)

## Old observation model

In [1]:
# Observational model OLD

clade = "A"
dataset = "Combined"
#or_data.loc[or_data['Clade'] == clade] # or_data['Collection'] == dataset

df = or_data[or_data["Label"] == f'{clade} ({dataset})']
df
n_obs = 104 # the same number as for the SIR simulator

theta_c_a = float(df["carriage_PP"]/(df["carriage_nonPP"] + df["carriage_PP"])) # carriage rate of c as from the data.
theta_c_0 = float(df["carriage_nonPP"]/(df["carriage_nonPP"] + df["carriage_PP"]))
theta_bsi_a = float(df["Disease_PP"]/(df["Disease_nonPP"] + df["Disease_PP"]))
theta_bsi_0 = float(df["Disease_nonPP"]/(df["Disease_nonPP"] + df["Disease_PP"]))

OR_hat = np.random.normal(0.4870, 0.1**2, n_obs)
plt.hist(OR_hat)
plt.title("OR_hat")
plt.show()


theta_c_a_obs = theta_bsi_a*theta_c_0/(OR_hat*theta_bsi_0) # this is now I_obs!
theta_c_0_obs = OR_hat*theta_c_a*theta_bsi_0/theta_bsi_a # This is S_obs!

# I can probably safely discard R from the equations altogether, since it is completely defined by S and I anyways.


def obsSI(or_data, clade, dataset, n_obs):
    
    # choose the clade and dataset
    df = or_data[or_data["Label"] == f'{clade} ({dataset})']
    
    # relevant quantities from data
    theta_c_a = float(df["carriage_PP"]/(df["carriage_nonPP"] + df["carriage_PP"])) # carriage rate of c as from the data.
    theta_c_0 = float(df["carriage_nonPP"]/(df["carriage_nonPP"] + df["carriage_PP"]))
    theta_bsi_a = float(df["Disease_PP"]/(df["Disease_nonPP"] + df["Disease_PP"]))
    theta_bsi_0 = float(df["Disease_nonPP"]/(df["Disease_nonPP"] + df["Disease_PP"]))

    # odds ratios from a normal distribution for a bit of uncertainty (based on data)
    or_mu = df["OR"]
    or_sd = (df["upper"] - df["lower"])/2
    OR_hat = np.random.normal(or_mu, or_sd**2, n_obs)
    
    # get the relevant observed quantities
    theta_c_a_obs = theta_bsi_a*theta_c_0/(OR_hat*theta_bsi_0) # this is now I_obs!
    theta_c_0_obs = OR_hat*theta_c_a*theta_bsi_0/theta_bsi_a # This is S_obs!
    
    return np.array([theta_c_0_obs]), np.array([theta_c_a_obs]) # same as S (not colonized by clade of interest) and I (colonized by clade of interest) compartment from the simulation

ds = "NORM"
N = 104
obsA = obsSI(or_data, clade = "A", dataset = ds, n_obs = N)
obsB = obsSI(or_data, clade = "B", dataset = ds, n_obs = N)
obsC1 = obsSI(or_data, clade = "C1", dataset = ds, n_obs = N)
obsC2 = obsSI(or_data, clade = "C2", dataset = ds, n_obs = N)

print(f"Observational model, dataset: {ds}")
print("Theta_c_clade")
f, axs = plt.subplots(2, 2, sharey = True, sharex = True)
axs[0,0].hist(obsA[1][0])
axs[0,0].set_title('Clade A')
axs[0,1].hist(obsB[1][0])
axs[0,1].set_title('Clade B')
axs[1,0].hist(obsC1[1][0])
axs[1,0].set_title('Clade C1')
axs[1,1].hist(obsC2[1][0])
axs[1,1].set_title('Clade C2')


NameError: name 'or_data' is not defined

# Custom discrepancy measure for a more complex ELFI configuration

Does not solve dimension issues with multiple nodes :(

In [ ]:
# A custom discrepancy measure between obs and simulated summaries
# Currently not necessary

SIR_obs = prop_to_nSIR(propSIR_simulator(0.734, 1/30, nt = n_weeks, N = pop_size, batch_size = 1), pop_size)
OR_hat = get_OR_hat(or_data, clade = "A", dataset = "NORM")
bsi_obs = sum_over_bsi(col_to_BSI(SIR_obs, OR_hat = OR_hat, is_prop = False))#.flatten()


SIR_sim = prop_to_nSIR(propSIR_simulator(0.7, 1/20, nt = n_weeks, N = pop_size, batch_size = 3), pop_size)
OR_hat = get_OR_hat(or_data, clade = "A", dataset = "NORM")
bsi_sim = sum_over_bsi(col_to_BSI(SIR_sim, OR_hat = OR_hat, is_prop = False))#.flatten()
print(bsi_sim.shape)


# summaries

s1 = BSI_max_t(bsi_sim)
s2 = BSI_max(bsi_sim)

obsS1 = BSI_max_t(bsi_obs)
obsS2 = BSI_max(bsi_obs)

print(s1.shape)
print(obsS1.shape)

print(np.array([obsS1, obsS2]).reshape(-1,1).transpose().shape)

observed = (obsS1, obsS2)

print(np.column_stack((s1,s2)).shape)
print(scipy.spatial.distance.cdist(np.column_stack((s1,s2)), np.array([observed[0], observed[1]]).transpose()))
print(scipy.spatial.distance.cdist(np.column_stack((s1,s2)), np.array([observed[0], observed[1]]).transpose()).shape)


def euclidean_distance(s1, s2, observed):
    # Does not help
    print(s1)
    print(s2)
    print(observed)
    
    return scipy.spatial.distance.cdist(np.column_stack((s1,s2)), np.array([observed[0], observed[1]]).transpose())



# Random stuff

In [1]:
    if is_bsi:
        
        or_data = bsi_pars["or_data"]
        clade = bsi_pars["clade"]
        dataset = bsi_pars["dataset"]
        theta_c = bsi_pars["theta_c"]
        theta_bsi = bsi_pars["theta_bsi"]
        
        
        or_hat = get_OR_hat(or_data = or_data, clade = clade, dataset = dataset, batch_size = batch_size, random_state = random_state)
        bsi = col_to_BSI((thetaS, thetaI, thetaR), OR_hat, theta_c = 1, theta_bsi = 0.3)
        #print(len(bsi))
        
        if agg_bsi:
            bsi = aggregate_BSI(bsi, nan_locations = nan_locations, batch_size = batch_size)
            return bsi, bsi, bsi
        
        return bsi

SyntaxError: 'return' outside function (<ipython-input-1-8988159111e7>, line 16)

# Data related

In [2]:
# Deprecated, theta_BSI is grouped by age this time
def get_theta_BSI(df, pop_data, as_prop = True):
    # df: NORM or BSAC data
    # pop_data: yearly population counts
    
    bsi_counts = np.flip(pd.value_counts(df["year"]))
    pop_counts = np.transpose(pop_data)

    pop_counts["prop_BSI"] = np.array(bsi_counts)/np.array(pop_counts["Population 1 January"])
    
    return pop_counts

get_theta_BSI(norm_data, norway_pop_data)


def insert_missing_data(bsi_obs):
    # Insert missing data to observed bsi as follows:
    # - If between two years, take the average number of cases in these years and use it as the value for the missing year
    # - If the last value, then take the previous value and use it as the missing value
    # - If the first value is missing, take the next value and use it as the missing value
    # - Deprecated - missing values are now handled by the get_obs_BSI function.
    
    obs_data = dict()
    for year in range(min(bsi_obs.index), max(bsi_obs.index) + 1):

        try:
            obs_data[year] = bsi_obs[year]
        except KeyError:
            #obs_data[year] = np.nan
            # note: cannot handle nan's for next and previous values
            if year != max(bsi_obs.index) and year != min(bsi_obs.index): # average of the previous and next value
                obs_data[year] = int((bsi_obs[year - 1] + bsi_obs[year + 1])/2)
            elif year == max(bsi_obs.index):
                obs_data[year] = bsi_obs[year - 1]
            elif year == min(bsi_obs.index):
                obs_data[year] = bsi_obs[year + 1]

    bsi_obs = np.asarray(list(obs_data.values()))

    return bsi_obs


NameError: name 'norm_data' is not defined

# SIR simulation grid



In [ ]:
def hide_ticks(ax_get_tick_labels, nth):
    # Based on: https://stackoverflow.com/questions/6682784/reducing-number-of-plot-ticks
    # ax_get_tick_labels: ax.xaxis.get_ticklabels() or ax.yaxis.get_ticklabels()
    # nth: which labels are shown?
    
    for n, label in enumerate(ax_get_tick_labels):
        if n % nth != 0:
            label.set_visible(False)
    
    
def plot_heatmap(mat, betas, gammas, title, cbar_label, nth = 4):
    # mat: matrix of interest, approximate likelihoods or distances for example
    # betas: row labels (parameter values at the row)
    # gammas: col labels (parameter values)
    # title: title of the plot
    # cbar_label: color bar label
    # nth: how "manyeth" tick label is displayed
    
    fig, ax = plt.subplots(figsize=(10, 10))

    row_labels = np.round(betas, 2)
    col_labels = np.round(gammas,2)

    im = ax.imshow(mat)
    ax.set_yticks(np.arange(mat.shape[0]), labels=row_labels)
    ax.set_xticks(np.arange(mat.shape[1]), labels=col_labels, rotation = 45)
    ax.set_xlabel("Gamma")
    ax.set_ylabel("Beta")
    
    cbar = ax.figure.colorbar(im, ax=ax)
    cbar.ax.set_ylabel(cbar_label, rotation=-90, va="bottom")
    ax.set_title(title)

    hide_ticks(ax.xaxis.get_ticklabels(), nth = nth)
    hide_ticks(ax.yaxis.get_ticklabels(), nth = nth)

    plt.show()

In [ ]:
# dimensions of the grid to simulate
n_beta = 300
n_gamma = 300

beta_range = (0.1, 1.0) # range of beta values to simulate
gamma_range = (0.0125, 0.1) # range of gamma values to simulate

def test_beta_and_gamma(beta, gamma):
    # returns False for the pair beta, gamma if they do not create an epidemic or R0 > 8.
    # returns True if beta/gamma is within the allowed range
    
    if beta/gamma <= 1:
        #print("No epidemic. Disregard this combo.")
        return False
        
    if beta/gamma > 8:
        #print("Infeasible parameters, R0 > 8.")
        return False

    return True



def check_beta_gamma_pairs(betas, gammas):
    # Check for infeasible combos of beta and gamma:
    bad_combos = []
    for i in range(0, len(betas)):
        for j in range(0, len(gammas)):
            if not test_beta_and_gamma(betas[i], gammas[j]):
                bad_combos.append((betas[i], gammas[j]))

    print(f"Simulated {i*j} pairs of parameters, {len(bad_combos)} of which were not within 1 < R0 < 8.")
    
    return bad_combos

def get_distance_grid(beta_range, gamma_range, n_beta, n_gamma, dist_func, summaries, bsi_obs, is_agg = False, scale = True):
    # Get the distance between pairs of gamma and beta in the given range
    
    betas = np.linspace(beta_range[0], beta_range[1], n_beta)
    gammas = np.linspace(gamma_range[0], gamma_range[1], n_gamma)

    # Save the squared differences between true and simulated summaries
    #delta_S1_mat = np.zeros((n_beta, n_gamma))
    #delta_S2_mat = np.zeros((n_beta, n_gamma))
    
    summary_mat = np.zeros((n_beta, n_gamma, len(summaries)))
    #summary_mat.fill(2)
    summary_mat.fill(np.nan)
    
    dist_mat = np.zeros((n_beta, n_gamma))
    #dist_mat.fill(10)
    #dist_mat = np.empty((n_beta, n_gamma))
    dist_mat.fill(np.nan)
    
    iters = 0

    for i in range(0, n_beta):
        for j in range(0, n_gamma):

            if iters%1000 == 0:
                print("Iter:", iters)
                
            beta = betas[i]
            gamma = gammas[j]
            
            if test_beta_and_gamma(beta, gamma):

                sim_seq = SIR_and_BSI_simulator(beta, gamma, n_weeks, pop_size, bsi_pars,\
                                      is_prop = True, is_agg = is_agg, time_period = 52, reparam = False,\
                                      batch_size = 1, random_state = None)[0]

                k = 0
                for summary in summaries:
                    summary_mat[i,j,k] = (summary(bsi_obs) - summary(sim_seq))**2 # to be fair, this is a distance matrix for a summary, not a summary matrix
                    k += 1
                
            #delta_S1_mat[i,j] = (BSI_max(bsi_obs) - BSI_max(sim_seq))**2
            #delta_S2_mat[i,j] = (BSI_max_t(bsi_obs) - BSI_max_t(sim_seq))**2
            
                dist = dist_func(sim_seq, bsi_obs, summaries) # redundant? unless not using scaling

                if not isinstance(dist, float):
                    print(f"Warning! Not float. Iter {iters}.")
                    print(dist)
                    print(f"Parameter pair: {beta}, {gamma}")
                else:
                    dist_mat[i,j] = dist

            iters += 1
            
    if scale:
        
        summary_mat = np.ma.array(summary_mat, mask = np.isnan(summary_mat)) # mask the nan values so that np.std does not return nans
        
        SD1 = np.std(summary_mat[:,:,0])
        
        dist_mat = 1/SD1*summary_mat[:,:,0]
        
        for k in range(1, len(summaries)):
            
            SD = np.std(summary_mat[:,:,k])
            
            if SD == 0:
                print("Warning! SD is zero. Summary index:", k)
                SD = 1 # TODO: what to do in this case? 
                
            dist_mat += 1/SD*summary_mat[:,:,k]
            
        dist_mat = np.sqrt(dist_mat)
    
    #return dist_mat, betas, gammas, delta_S1_mat, delta_S2_mat# return betas and gammas for plotting purposes
    
    return dist_mat, betas, gammas, summary_mat


dist_mat, betas, gammas, summary_mat = get_distance_grid(beta_range, gamma_range, n_beta, n_gamma,\
                                                         distance_generalized, [BSI_max, BSI_max_t], bsi_obs[0])
#dist_mat, betas, gammas, summary_mat = get_distance_grid(beta_range, gamma_range, n_beta, n_gamma, bsi_obs[0])
plot_heatmap(dist_mat, betas, gammas, title = "Distance(S(obs), S(sim))", cbar_label = "Distance", nth = 10) # synthetic observed data

# Q: what happens to the BSI_max summary?
# TODO: better alternative is to sample from a range of beta, gamma values that is always ok.
plot_heatmap(summary_mat[:,:,0], betas, gammas, title = "(S1_obs - S1_sim)^2", cbar_label = "Discrepancy (BSI_max)", nth = 8)
# Works, if I set summary_mat to nan. Probably a scaling issue if I set it to anything else.
        
    

In [ ]:
# Approximate likelihood
# Step 1: Approximation of the likelihood function


def approx_likelihood(dist, epsilon=0.0001):
    # Calculates the approximate likelihood over N simulations for a given parameter pair
    # dist: distances between the simulated and observed sequences over N simulations
    
    N = dist.shape[0]
    lik = 0
    for d in dist:
        #print(d)
        
        if d < epsilon:
            lik += d
    
    return 1/N*lik

    
# Step 3: Running the simulation over pairs of parameters and get the approximate likelihood

# Pick parameters for the simulation

# dimensions of the grid to simulate
n_beta = 50
n_gamma = 40

N_sim = 100 # number of simulations at each point in the grid
e = 0.01 # epsilon, large enough epsilon means that all distances will be in the grid

beta_range = (0.5, 1.5) # range of beta values to simulate
gamma_range = (0.0125, 0.1) # range of gamma values to simulate

def approx_likelihood_grid(beta_range, gamma_range, n_beta, n_gamma, N_sim, e, bsi_obs):
    betas = np.linspace(beta_range[0], beta_range[1], n_beta)
    gammas = np.linspace(gamma_range[0], gamma_range[1], n_gamma)

    approx_lik_mat= np.zeros((n_beta, n_gamma))
    iters = 0
    for i in range(0, n_beta):
        for j in range(0, n_gamma):

            if iters%100 == 0:
                print("Iter:", iters)
            beta = betas[i]
            gamma = gammas[j]

            distances = np.zeros(N_sim)

            for h in range(0, N_sim):
                sim_seq = SIR_and_BSI_simulator(beta, gamma, n_weeks, pop_size, bsi_pars,\
                                  is_prop = True, is_agg = False, time_period = 52, reparam = False,\
                                  batch_size = 1, random_state = None)[0]
                distances[h] = distance(sim_seq, bsi_obs, BSI_max, BSI_max_t)

            # Calculate the approximate likelihood for this parameter pair:

            L_hat = approx_likelihood(distances, epsilon=e)
            if not isinstance(L_hat, float):
                print(f"Warning! Not float. Iter {iters}.")
                print(L_hat)
                print(f"Parameter pair: {beta}, {gamma}")
            else:
                approx_lik_mat[i,j] = L_hat

            iters += 1
    
    return approx_lik_mat, betas, gammas

#approx_lik_mat, betas, gammas = approx_likelihood_grid(beta_range, gamma_range, n_beta, n_gamma, N_sim, e, bsi_obs[0])

In [ ]:
# Using the new t0 summary statistic:
dist_mat, betas, gammas, summary_mat = get_distance_grid(beta_range, gamma_range, n_beta, n_gamma,\
                                                         distance_generalized, [BSI_max, BSI_max_t, BSI_delta_t, BSI_k, BSI_var], bsi_obs[0])

In [ ]:
# Grid simulation
# dimensions of the grid to simulate
n_beta = 100
n_gamma = 90

beta_range = (0.01, 0.7) # range of beta values to simulate; has to be large enough to find the small distance areas,
# which can then be used to refine this range further. And used as values for synthetic data.
gamma_range = (0.0125, 0.5) # range of gamma values to simulate

dist_mat, betas, gammas, delta_S1_mat, delta_S2_mat = get_distance_grid(beta_range, gamma_range, n_beta, n_gamma, bsi_obs_data, is_agg = True)

# plottaa summaryitä näistä pisteistä: erikseen ja yhdessä molemmat

plot_heatmap(dist_mat, betas, gammas, title = "Distance(S(obs), S(sim))", cbar_label = "Distance", nth = 8) # sus